In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from collections import defaultdict, Counter
from sklearn.utils import resample
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, mean_squared_error
from sklearn.model_selection import train_test_split
from scipy.special import expit
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
# File path
notes_file = os.path.join(DATA_DIR, 'NOTEEVENTS.csv')
diagnosis_file = os.path.join(DATA_DIR, 'DIAGNOSES_ICD.csv')
icd_desc_file = os.path.join(DATA_DIR, 'D_ICD_DIAGNOSES.csv')

# Step 1: read the discharge notes
def load_discharge_summaries(file_path):
    print("Loading discharge summaries...")
    df = pd.read_csv(file_path, low_memory=False)
    discharge_df = df[df['CATEGORY'] == 'Discharge summary']
    discharge_df = discharge_df.sort_values(['HADM_ID', 'CHARTDATE']).drop_duplicates('HADM_ID', keep='last')
    return discharge_df[['SUBJECT_ID', 'HADM_ID', 'TEXT']]

discharge_df = load_discharge_summaries(notes_file)
print(f"Discharge summaries: {discharge_df.shape}")

# Step 2: ICD labels processing
def load_icd_labels(diag_file):
    print("Loading ICD codes...")
    diag_df = pd.read_csv(diag_file)
    grouped = diag_df.groupby('HADM_ID')['ICD9_CODE'].apply(list).reset_index()
    return grouped

icd_df = load_icd_labels(diagnosis_file)
print(f"ICD labels: {icd_df.shape}")


# Step 3: Link text and label
print("Merging text and labels...")
data = pd.merge(discharge_df, icd_df, on='HADM_ID')
data = data.dropna(subset=['TEXT', 'ICD9_CODE'])

# Step 3.5: Standardization
def ensure_text(x):
    return str(x) if not isinstance(x, str) else x

def ensure_list_of_str(x):
    if isinstance(x, str):
        return [x]
    return list(map(str, x))

data['TEXT'] = data['TEXT'].apply(ensure_text)
data['ICD9_CODE'] = data['ICD9_CODE'].apply(ensure_list_of_str)
print(f"ICD labels: {icd_df.shape}")

assert isinstance(data['TEXT'].iloc[0], str)
assert isinstance(data['ICD9_CODE'].iloc[0], list)
assert isinstance(data['ICD9_CODE'].iloc[0][0], str)


# Step 4: save as pickle
data.to_pickle('./mimic3_data_test.pkl')
print(f"Saved merged dataset to {output_path}")


Loading discharge summaries...
Discharge summaries: (52726, 3)
Loading ICD codes...
ICD labels: (58976, 2)
Merging text and labels...
ICD labels: (58976, 2)
Saved merged dataset to /content/drive/MyDrive/CMU/02750/mimic3_data_test.pkl
